# Rodinia GPU traces on Colab → Nsight Systems on your laptop
Select **Runtime → Change runtime type → GPU**, then run cells in order.

This notebook collects traces **while the workloads execute on Colab**. Your laptop opens the resulting `.nsys-rep`; it cannot retrospectively trace a completed run. We also export SQLite for later DAG construction.

Uses the repository's unchanged Rodinia Nearest Neighbor (513 records) and HotSpot3D (64×64×3, 2 iterations), matching our small Clang scenarios. Inputs are deterministic **synthetic data**, not the official benchmark dataset. These small cases validate structure, not representative performance.

HotSpot3D's upstream code downloads the previous iteration's buffer. We preserve this behavior for comparison; its printed Accuracy is not a reliable certification of the latest output. No numerical-correctness claim is made by this notebook.

No internal GPU hardware counters are requested. Colab may restrict CUPTI tracing; a smoke test checks actual GPU events before proceeding. If a cell fails, run the final download cell to retain diagnostics.

References: [Nsight installation](https://docs.nvidia.com/nsight-systems/InstallationGuide/) · [CLI guide](https://docs.nvidia.com/nsight-systems/UserGuide/).

In [ ]:
from pathlib import Path
import subprocess, shutil, json, hashlib, datetime, sqlite3, os, re
BASE = Path('/content/rodinia_capture')
BASE.mkdir(parents=True, exist_ok=True)
RUN = BASE / datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
RUN.mkdir()
LOGS = RUN / 'logs'; LOGS.mkdir()
def run(args, name, cwd=None, check=True):
    args = [str(x) for x in args]
    p = subprocess.run(args, cwd=cwd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    (LOGS / (name + '.log')).write_text('$ ' + repr(args) + '\n' + p.stdout)
    print(p.stdout[-12000:])
    if check and p.returncode:
        raise RuntimeError(f'{name} failed ({p.returncode}); see logs and run the final download cell.')
    return p
assert shutil.which('nvidia-smi'), 'No NVIDIA runtime detected. Select Runtime → Change runtime type → GPU, reconnect, and rerun from the first cell.'
run(['nvidia-smi'], 'gpu')
assert shutil.which('nvcc'), 'CUDA compiler missing: select a Colab GPU runtime.'
run(['nvcc', '--version'], 'nvcc')
print('Results:', RUN)

## Install or locate the collector
Uses the installed collector if available. Otherwise installs NVIDIA's CLI package using a signed NVIDIA repository on Ubuntu. It does not replace the GPU driver. Use a laptop Nsight Systems version compatible with the recorded collector version (prefer the same release).

In [ ]:
def locate_nsys():
    found = shutil.which('nsys')
    if found: return found
    candidates = list(Path('/opt/nvidia').glob('nsight-systems*/**/nsys'))
    candidates += list(Path('/usr/local').glob('cuda*/bin/nsys'))
    return str(sorted(candidates)[-1]) if candidates else None
NSYS = locate_nsys()
if not NSYS:
    run(['apt-get', 'update', '-qq'], 'apt_update')
    run(['apt-get', 'install', '-y', 'wget', 'gnupg', 'ca-certificates'], 'apt_dependencies')
    release = dict(line.split('=',1) for line in Path('/etc/os-release').read_text().splitlines() if '=' in line)
    assert release['ID'].strip('"') == 'ubuntu', 'Automatic installation supports Ubuntu only.'
    version = release['VERSION_ID'].strip('"').replace('.', '')
    arch = subprocess.check_output(['dpkg','--print-architecture'],text=True).strip()
    assert arch == 'amd64', 'This notebook targets Colab x86_64.'
    run(['wget','-q','-O','/tmp/nvidia-devtools.pub',
         'https://developer.download.nvidia.com/compute/cuda/repos/ubuntu1804/x86_64/7fa2af80.pub'], 'key_download')
    run(['gpg','--batch','--yes','--dearmor','-o','/usr/share/keyrings/nvidia-devtools-keyring.gpg','/tmp/nvidia-devtools.pub'], 'key_install')
    Path('/etc/apt/sources.list.d/nvidia-devtools.list').write_text(
        f'deb [signed-by=/usr/share/keyrings/nvidia-devtools-keyring.gpg] https://developer.download.nvidia.com/devtools/repos/ubuntu{version}/{arch}/ /\n')
    run(['apt-get','update','-qq'], 'devtools_update')
    run(['apt-get','install','-y','nsight-systems-cli'], 'collector_install')
    NSYS = locate_nsys()
assert NSYS, 'Collector not found. Download the diagnostics; do not treat an ordinary run as a trace.'
run([NSYS,'--version'], 'nsys_version')
run([NSYS,'status','--environment'], 'environment', check=False)
# CPU sampling restrictions are not decisive: the next cell tests CUDA tracing itself.

## Fetch a pinned source revision and compile
We pin the source revision to make comparison repeatable. Compilation adds standard-library declarations via `-include`; CUDA workload source files are not rewritten. A compiler definition maps the legacy host API `cudaThreadSynchronize` to `cudaDeviceSynchronize`, preserving the device-wide wait used by Nearest Neighbor. The flag is recorded in the manifest. The selected GPU's compute capability determines the target architecture.

In [ ]:
REPO = BASE / 'ESP_workload'
REVISION = '4106565d567f39851e0371ab07a4efdeaf3fd3d2'
if not REPO.exists():
    run(['git','clone','https://github.com/pa2746-byte/ESP_workload.git',REPO], 'clone')
run(['git','-C',REPO,'checkout','--detach',REVISION], 'checkout')
cap = run(['nvidia-smi','--query-gpu=compute_cap','--format=csv,noheader'], 'compute_cap').stdout.splitlines()[0].strip().replace('.','')
assert cap.isdigit(), f'Unexpected compute capability: {cap}'
BUILD = RUN / 'build'; BUILD.mkdir(exist_ok=True)
SOURCES = RUN / 'sources'; SOURCES.mkdir(exist_ok=True)
for name in ['nn','hotspot3d']:
    shutil.copytree(REPO/'workloads/rodinia'/name, SOURCES/name, dirs_exist_ok=True)
shutil.copy(REPO/'workloads/rodinia/LICENSE', SOURCES/'LICENSE')
FLAGS = ['-DcudaThreadSynchronize=cudaDeviceSynchronize', '-O3','-lineinfo','-std=c++14',f'-arch=sm_{cap}', '-include','cstring','-include','cstdlib','-include','cmath']
run(['nvcc',*FLAGS,SOURCES/'nn/nn_cuda.cu','-o',BUILD/'nn'], 'build_nn')
run(['nvcc',*FLAGS,SOURCES/'hotspot3d/3D.cu','-o',BUILD/'hotspot3d'], 'build_hotspot3d')
# Preserve existing Clang outputs/configs for this exact small scenario.
for name in ['rodinia_nn','rodinia_hotspot3d']:
    shutil.copytree(REPO/'results'/name, RUN/'clang_reference'/name, dirs_exist_ok=True)

## Confirm GPU tracing works
The smoke program checks CUDA errors and its computed result. A report is accepted only when its SQLite export contains GPU kernel and memcpy activity. CPU API events alone are insufficient.

In [ ]:
smoke = BUILD / 'smoke.cu'
smoke.write_text(r"""
#include <cuda_runtime.h>
#include <cstdio>
#define CHECK(call) do { cudaError_t e=(call); if(e!=cudaSuccess) { fprintf(stderr,"%s\n",cudaGetErrorString(e)); return 1; } } while(0)
__global__ void increment(int *p) { *p += 1; }
int main(){int h=41; int *d; CHECK(cudaMalloc(&d,sizeof(int)));
CHECK(cudaMemcpy(d,&h,sizeof(int),cudaMemcpyHostToDevice));
increment<<<1,1>>>(d); CHECK(cudaGetLastError()); CHECK(cudaDeviceSynchronize());
CHECK(cudaMemcpy(&h,d,sizeof(int),cudaMemcpyDeviceToHost)); CHECK(cudaFree(d));
return h==42 ? 0 : 2;}
""")
run(['nvcc',*FLAGS,smoke,'-o',BUILD/'smoke'], 'build_smoke')
def inspect_trace(db):
    with sqlite3.connect(f'file:{db}?mode=ro', uri=True) as con:
        tables = {r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")}
        kernels = [t for t in tables if t.startswith('CUPTI_ACTIVITY_KIND_') and 'KERNEL' in t]
        copies = [t for t in tables if t == 'CUPTI_ACTIVITY_KIND_MEMCPY']
        counts = {t:con.execute('SELECT COUNT(*) FROM "'+t+'"').fetchone()[0] for t in kernels+copies}
        assert sum(counts[t] for t in kernels)>0, 'No GPU kernel events: capture unsupported/incomplete.'
        assert sum(counts[t] for t in copies)>0, 'No GPU memcpy events: capture incomplete.'
        return counts
TRACE_STATUS = {}
def profile(name, command, cwd=None):
    prefix = RUN / name
    run([NSYS,'profile','--trace=cuda','--sample=none','--cpuctxsw=none',
         '--force-overwrite=true','--output',prefix,*command], 'profile_'+name, cwd=cwd)
    report = prefix.with_suffix('.nsys-rep')
    assert report.exists() and report.stat().st_size, 'Missing imported report; retain .qdstrm and logs for diagnosis.'
    db = prefix.with_suffix('.sqlite')
    run([NSYS,'export','--type=sqlite','--force-overwrite=true','--output',db,report], 'export_'+name)
    counts = inspect_trace(db)
    TRACE_STATUS[name] = counts
    (RUN/'trace_validation.json').write_text(json.dumps(TRACE_STATUS,indent=2))
    print(name, counts)
    return db
profile('smoke',[BUILD/'smoke'])

## Create reproducible inputs
Nearest Neighbor's parser expects exactly 48 characters followed by a newline, with latitude at offset 28 and longitude at offset 33. HotSpot3D expects one scalar per line. Values are generated deterministically and archived with hashes.

In [ ]:
INPUTS = RUN / 'inputs'; INPUTS.mkdir(exist_ok=True)
records = []
for i in range(513):
    lat = 10.0 + (i % 70) * 0.5
    lng = 40.0 + (i % 90) * 0.5
    record = f'record{i:06d}'.ljust(28) + f'{lat:5.1f}{lng:5.1f}' + ' '*10
    assert len(record)==48
    records.append(record+'\n')
(INPUTS/'records.txt').write_text(''.join(records))
(INPUTS/'filelist.txt').write_text('records.txt\n')
size = 64*64*3
(INPUTS/'power.txt').write_text(''.join(f'{0.1+(i%17)*0.01:.6f}\n' for i in range(size)))
(INPUTS/'temperature.txt').write_text(''.join(f'{70+(i%23)*0.1:.6f}\n' for i in range(size)))
COMMANDS = {
    'nn': [str(BUILD/'nn'),'filelist.txt','-r','5','-lat','30','-lng','90','-q'],
    'hotspot3d': [str(BUILD/'hotspot3d'),'64','3','2','power.txt','temperature.txt',str(RUN/'hotspot3d_output.txt')]
}
manifest = dict(source_revision=REVISION, compile_flags=FLAGS, commands=COMMANDS,
    parameters=dict(nn_records=513, rows=64,layers=3,iterations=2),
    input_kind='deterministic synthetic data',
    caveats=['HotSpot3D preserves upstream previous-iteration download.',
             'Event checks validate trace structure, not numerical correctness or performance fidelity.'])
manifest['sha256'] = {str(p.relative_to(RUN)):hashlib.sha256(p.read_bytes()).hexdigest()
    for folder in (INPUTS,SOURCES) for p in folder.rglob('*') if p.is_file()}
(RUN/'manifest.json').write_text(json.dumps(manifest,indent=2))

## Run and capture both workloads
Each ordinary run is followed by a separate profiled process. These are not steady-state timing benchmarks. We verify kernel counts and explicit copy volumes for the pinned scenarios. We do not assert kernel logical footprints equal measured DRAM traffic.

In [ ]:
EXPECTED = {'nn': (1, [2052,4104]), 'hotspot3d': (2,[49152,49152,49152])}
for name,command in COMMANDS.items():
    run(command, 'run_'+name, cwd=INPUTS)
    db = profile(name,command,cwd=INPUTS)
    with sqlite3.connect(f'file:{db}?mode=ro',uri=True) as con:
        tables = [r[0] for r in con.execute("SELECT name FROM sqlite_master WHERE type='table'")]
        kernels = [t for t in tables if t.startswith('CUPTI_ACTIVITY_KIND_') and 'KERNEL' in t]
        nk = sum(con.execute('SELECT COUNT(*) FROM "'+t+'"').fetchone()[0] for t in kernels)
        volumes = sorted(r[0] for r in con.execute('SELECT bytes FROM CUPTI_ACTIVITY_KIND_MEMCPY'))
    assert (nk,volumes)==EXPECTED[name], f'{name}: unexpected events {nk}, {volumes}; inspect trace before comparison.'
    print(name, 'PASS: expected kernel count and copy byte counts. Numerical correctness is not certified.')

## Download the bundle (also run this after a failure)
Open `nn.nsys-rep` and `hotspot3d.nsys-rep` using **File → Open** in your laptop's Nsight Systems. Expand CUDA GPU activity to inspect kernels and memory copies. If the reader rejects a newer report, use a matching supported desktop release.

Keep the SQLite files, logs, source files, manifest, inputs, and Clang reference graphs together. Upload the ZIP back to this conversation for DAG construction/comparison. This notebook does not convert timestamps into dependency edges: stream order and synchronization must be handled explicitly.

A ZIP containing only logs or `.qdstrm` is a diagnostic bundle, not proof of successful trace collection. Colab storage is temporary; save the ZIP before disconnecting.

In [ ]:
archive = shutil.make_archive(str(RUN), 'zip', root_dir=RUN)
print('Bundle:', archive)
print('Validated reports:', sorted(globals().get('TRACE_STATUS', {})))
from google.colab import files
files.download(archive)